In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from ugdatalab.models.galaxy_zoo import GalaxyZooDataset
from ugdatalab.models.galaxy_zoo.constants import N_LABELS
from ugdatalab.methods.cnn import train_cnn, count_parameters
from ugdatalab.methods.architectures import build_custom_cnn

import plotters

# Galaxy Image Classification — Custom CNN

## Task 16 — Custom CNN Architecture

We build a custom CNN following the lab manual guidelines: convolution layers with pooling, followed by fully connected layers with dropout, ending in a sigmoid activation to constrain outputs to $[0, 1]$.

**Architecture choices:**
- **3 convolutional blocks** (32 → 64 → 128 channels) with max pooling after each. The increasing channel depth allows the network to learn progressively more abstract features: edges → textures → morphological components.
- **Kernel sizes** of 5 for the first layer (captures larger-scale features at the input resolution) and 3 for deeper layers (standard choice once spatial dimensions are reduced).
- **2 fully connected layers** (256, 128 units) with ReLU activation and 50% dropout. Dropout is the primary regularization mechanism, preventing the FC layers from memorizing training examples.
- **Adam optimizer** with initial learning rate $10^{-3}$.
- **Sigmoid output** ensures all 37 predictions lie in $[0, 1]$, matching the label range.

Target: RMSE $\leq 0.11$.

In [ ]:
# Load preprocessed data
img_data = np.load("galaxy_zoo_images.npz")
images = img_data["images"]
label_data = np.load("galaxy_zoo_labels.npz")
labels = label_data["labels"]
split_data = np.load("split_indices.npz")
train_idx, val_idx = split_data["train_idx"], split_data["val_idx"]

train_images, val_images = images[train_idx], images[val_idx]
train_labels, val_labels = labels[train_idx], labels[val_idx]
TARGET_SIZE = images.shape[1]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

train_ds = GalaxyZooDataset(train_images, train_labels, transform=None)
val_ds = GalaxyZooDataset(val_images, val_labels, transform=None)

In [ ]:
model = build_custom_cnn(
    n_labels=N_LABELS,
    n_channels_list=[32, 64, 128],
    kernel_sizes=[5, 3, 3],
    fc_sizes=[256, 128],
    dropout_rate=0.5,
    pool_type="max",
    input_size=TARGET_SIZE,
)
print(model)
print(f"\nCustom CNN trainable parameters: {count_parameters(model):,}")

In [ ]:
custom_result = train_cnn(
    model=model,
    train_dataset=train_ds,
    val_dataset=val_ds,
    batch_size=64,
    n_epochs=30,
    lr=1e-3,
    device=DEVICE,
    seed=42,
    scheduler_factory=None,
    num_workers=4,
)
print(f"\nBest epoch: {custom_result.best_epoch + 1}")
print(f"Best validation RMSE: {custom_result.best_val_loss:.4f}")

In [ ]:
ax = plotters.plot_loss_curves(
    custom_result.train_losses, custom_result.val_losses, "Custom CNN",
)
plt.show()

## Task 18 — Parameter Count Comparison

We compare the number of trainable parameters in our custom CNN vs. ResNet-18, and compare both to the total number of pixels in the compressed training set. If the number of parameters greatly exceeds the number of training pixels, the model has more capacity than the data can constrain, which should make us cautious about overfitting.

In [ ]:
import pandas as pd

resnet_data = np.load("resnet_result.npz")
resnet_params = int(resnet_data["n_parameters"])
custom_params = custom_result.n_parameters
total_pixels = train_images.shape[0] * TARGET_SIZE * TARGET_SIZE * 3

comparison = pd.DataFrame({
    "Model": ["Custom CNN", "ResNet-18", "Training pixels"],
    "Count": [f"{custom_params:,}", f"{resnet_params:,}", f"{total_pixels:,}"],
})
print(comparison.to_string(index=False))
print(f"\nCustom CNN / training pixels ratio: {custom_params / total_pixels:.2f}")
print(f"ResNet-18 / training pixels ratio: {resnet_params / total_pixels:.2f}")

## Task 19 — Model Selection

We compare the best validation RMSE from both models and select the better-performing one for optimization in NB 05.

In [ ]:
resnet_best = float(resnet_data["best_val_loss"])
custom_best = custom_result.best_val_loss

print(f"ResNet-18 best val RMSE: {resnet_best:.4f}")
print(f"Custom CNN best val RMSE: {custom_best:.4f}")

best_model = "resnet" if resnet_best < custom_best else "custom"
print(f"\nBest model: {'ResNet-18' if best_model == 'resnet' else 'Custom CNN'}")

# Save custom CNN results
torch.save(custom_result.model_state, "custom_cnn.pt")
np.savez_compressed(
    "custom_result.npz",
    train_losses=custom_result.train_losses,
    val_losses=custom_result.val_losses,
    best_epoch=custom_result.best_epoch,
    best_val_loss=custom_result.best_val_loss,
    n_parameters=custom_result.n_parameters,
    learning_rates=custom_result.learning_rates,
    best_model=best_model,
)
print("Saved custom_cnn.pt and custom_result.npz")